# Experiment 12: Hyperparameter Search with Optuna (Bayesian + ASHA Pruning)

**Single variable changed**: this is a new *phase*, not a single-variable ablation. All prior experiments (E1–E11) used fixed HPs. This phase finds the optimal HP configuration via Bayesian search, then re-evaluates top prior experiments against the tuned baseline.

## Search space

| Hyperparameter | Distribution | Rationale |
|---------------|-------------|-----------|
| Learning rate | log-uniform [1e-4, 1e-2] | Widest-impact knob; fixed 0.001 was never verified optimal |
| Dropout | uniform [0.2, 0.5] | Only tried 0.3 (E1–E9) and 0.5 (E2, confounded) |
| Weight decay | log-uniform [1e-6, 1e-3] | Untested in E1–E11 |
| Batch size | categorical {64, 128, 256} | Always 64 in prior experiments |
| Optimizer | categorical {Adam, AdamW, SGD} | Adam only in prior experiments |
| LR schedule | categorical {None, StepLR, CosineAnnealing, ReduceLROnPlateau} | CosineLR (E8–E10) and ReduceLROnPlateau (E11) only |

## Methodology

- **Bayesian Optimization** via Optuna's TPESampler — builds a surrogate model of HP → val-accuracy
- **ASHA-style pruning** via MedianPruner — kills trials below median accuracy at same epoch
- **30 trials** × 35 epochs each, with a 10% validation holdout
- **Study saved to SQLite** for resume capability

In [1]:
import sys, os, warnings; sys.path.append('../..'); warnings.filterwarnings('ignore')
import optuna, torch, torch.nn as nn, torch.optim as optim, numpy as np
import torchvision.transforms as transforms
from optuna.pruners import MedianPruner; from optuna.samplers import TPESampler
from torch.optim.lr_scheduler import StepLR, CosineAnnealingLR, ReduceLROnPlateau
from torch.utils.data import DataLoader, random_split
from src.train_utils import train_one_epoch; from src.eval_utils import evaluate

N_TRIALS = 30; N_EPOCHS = 35; VAL_SPLIT = 0.1; SEED = 42
STUDY_DIR = '../outputs/error_analysis/optuna_study'; os.makedirs(STUDY_DIR, exist_ok=True)

device = ('mps' if torch.backends.mps.is_available()
          else 'cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}  |  Trials: {N_TRIALS}  |  Epochs/trial: {N_EPOCHS}')

Device: cuda  |  Trials: 30  |  Epochs/trial: 35


## Data — shared across all trials

In [2]:
transform = transforms.Compose([
    transforms.ToTensor(), transforms.Normalize((0.5,), (0.5,)),
])
full_train = __import__('torchvision').datasets.FashionMNIST(
    root='../data', train=True, download=True, transform=transform)
test_ds = __import__('torchvision').datasets.FashionMNIST(
    root='../data', train=False, download=True, transform=transform)

val_len = int(len(full_train) * VAL_SPLIT)
train_ds, val_ds = random_split(
    full_train, [len(full_train) - val_len, val_len],
    generator=torch.Generator().manual_seed(SEED))
print(f'Train: {len(train_ds)}  Val: {len(val_ds)}  Test: {len(test_ds)}')

Train: 54000  Val: 6000  Test: 10000


## Model — DiagnosticCNN with configurable dropout

In [3]:
class DiagnosticCNN(nn.Module):
    def __init__(self, dropout=0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(True),
            nn.Conv2d(32, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(True),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(True),
            nn.Conv2d(64, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(True),
            nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(True),
            nn.MaxPool2d(2),
            nn.AdaptiveAvgPool2d(1), nn.Flatten(),
            nn.Dropout(dropout), nn.Linear(128, 10),
        )
    def forward(self, x):
        return self.net(x)

## Objective function — one trial

In [4]:
def objective(trial):
    lr = trial.suggest_float('lr', 1e-4, 1e-2, log=True)
    dropout = trial.suggest_float('dropout', 0.2, 0.5)
    weight_decay = trial.suggest_float('weight_decay', 1e-6, 1e-3, log=True)
    batch_size = trial.suggest_categorical('batch_size', [64, 128, 256])
    opt_name = trial.suggest_categorical('optimizer', ['Adam', 'AdamW', 'SGD'])
    sched_name = trial.suggest_categorical('lr_schedule',
                                           ['None', 'StepLR', 'CosineAnnealingLR', 'ReduceLROnPlateau'])

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=256, shuffle=False)

    model = DiagnosticCNN(dropout=dropout).to(device)
    opt_cls = {'Adam': optim.Adam, 'AdamW': optim.AdamW,
               'SGD': lambda p, **kw: optim.SGD(p, momentum=0.9, nesterov=True, **kw)}[opt_name]
    optimizer = opt_cls(model.parameters(), lr=lr, weight_decay=weight_decay)
    criterion = nn.CrossEntropyLoss()

    if sched_name == 'StepLR':
        scheduler = StepLR(optimizer, step_size=max(1, N_EPOCHS // 3), gamma=0.1)
    elif sched_name == 'CosineAnnealingLR':
        scheduler = CosineAnnealingLR(optimizer, T_max=N_EPOCHS)
    elif sched_name == 'ReduceLROnPlateau':
        scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.1, patience=5)
    else:
        scheduler = None

    best_val_acc = 0.0
    for epoch in range(N_EPOCHS):
        train_loss = train_one_epoch(model, train_loader, criterion, optimizer, device)

        model.eval()
        correct = total = 0
        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                preds = model(images).argmax(dim=1)
                correct += (preds == labels).sum().item()
                total += labels.size(0)
        model.train()
        val_acc = correct / total

        if scheduler is not None:
            scheduler.step(train_loss if isinstance(scheduler, ReduceLROnPlateau) else None)

        trial.report(val_acc, epoch)
        if trial.should_prune():
            raise optuna.TrialPruned()
        best_val_acc = max(best_val_acc, val_acc)

    return best_val_acc

## Run Optuna study

In [5]:
study = optuna.create_study(
    direction='maximize',
    sampler=TPESampler(seed=SEED),
    pruner=MedianPruner(n_startup_trials=5, n_warmup_steps=3),
    study_name='fashion_mnist_diagnostic_cnn',
    storage=f'sqlite:///{STUDY_DIR}/optuna_study.db',
    load_if_exists=True,
)
study.optimize(objective, n_trials=N_TRIALS)

print(f'\nBest trial #{study.best_trial.number}  |  Val-acc: {study.best_value:.4f}')
for k, v in study.best_params.items():
    print(f'  {k}: {v}')

[I 2026-07-26 14:12:15,638] A new study created in RDB with name: fashion_mnist_diagnostic_cnn
[I 2026-07-26 14:24:19,255] Trial 0 finished with value: 0.9358333333333333 and parameters: {'lr': 0.0005611516415334506, 'dropout': 0.4852142919229748, 'weight_decay': 0.000157029708840554, 'batch_size': 64, 'optimizer': 'AdamW', 'lr_schedule': 'CosineAnnealingLR'}. Best is trial 0 with value: 0.9358333333333333.
[I 2026-07-26 14:33:04,241] Trial 1 finished with value: 0.9256666666666666 and parameters: {'lr': 0.00026587543983272726, 'dropout': 0.2545474901621302, 'weight_decay': 3.549878832196506e-06, 'batch_size': 128, 'optimizer': 'AdamW', 'lr_schedule': 'ReduceLROnPlateau'}. Best is trial 0 with value: 0.9358333333333333.
[I 2026-07-26 14:40:34,656] Trial 2 finished with value: 0.9018333333333334 and parameters: {'lr': 0.00025081156860452336, 'dropout': 0.3542703315240835, 'weight_decay': 5.987474910461405e-05, 'batch_size': 128, 'optimizer': 'SGD', 'lr_schedule': 'None'}. Best is trial 


Best trial #24  |  Val-acc: 0.9392
  lr: 0.003322336843334369
  dropout: 0.45355592625028196
  weight_decay: 1.386093872693893e-05
  batch_size: 64
  optimizer: AdamW
  lr_schedule: CosineAnnealingLR


## Retrain best config on full training set → test

In [6]:
bp = study.best_params
model = DiagnosticCNN(dropout=bp['dropout']).to(device)
opt_cls = {'Adam': optim.Adam, 'AdamW': optim.AdamW,
           'SGD': lambda p, **kw: optim.SGD(p, momentum=0.9, nesterov=True, **kw)}[bp['optimizer']]
optimizer = opt_cls(model.parameters(), lr=bp['lr'], weight_decay=bp['weight_decay'])

full_loader = DataLoader(full_train, batch_size=bp['batch_size'], shuffle=True)
for epoch in range(N_EPOCHS):
    train_one_epoch(model, full_loader, nn.CrossEntropyLoss(), optimizer, device)

test_acc = evaluate(model, DataLoader(test_ds, batch_size=256, shuffle=False), device)
print(f'Test accuracy with best HPs: {test_acc:.2f}%')

Test Accuracy: 93.14%
Test accuracy with best HPs: 93.14%


## Hyperparameter importance analysis

In [7]:
try:
    import optuna.importance
    imp = optuna.importance.get_param_importances(study)
    print('Hyperparameter importance:')
    for k, v in sorted(imp.items(), key=lambda x: -x[1]):
        print(f'  {k:<20}  {v:.4f}')
except Exception as e:
    print(f'Importance analysis unavailable: {e}')

Hyperparameter importance:
  lr_schedule           0.3772
  batch_size            0.1705
  lr                    0.1595
  dropout               0.1303
  optimizer             0.1277
  weight_decay          0.0348


## Save results to disk

In [8]:
import json
with open(os.path.join(STUDY_DIR, 'best_params.json'), 'w') as f:
    json.dump({**bp, 'test_accuracy': test_acc, 'best_val_accuracy': study.best_value, 'n_trials': len(study.trials)}, f, indent=2)
print(f'Results saved to {STUDY_DIR}/')
print(f'Resume:  study.optimize(objective, n_trials=N_TRIALS + 30)')

Results saved to ../outputs/error_analysis/optuna_study/
Resume:  study.optimize(objective, n_trials=N_TRIALS + 30)
